In [1]:
import importlib
import numpy as np
import polars as pl
import scipy.sparse as sp
import torch

from datasets import DATA_FOLDER, prepare_interaction_data
from util import CHECKPOINT_FOLDER, get_checkpoint_filepath, load_checkpoint, load_config_from_checkpoint

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if torch.mps.is_available() else torch.device("cpu")

DATASET = "ml-25m"
SAE_CHECKPOINT_PATH = f"{CHECKPOINT_FOLDER}/{DATASET}/TopKSAE-8192-1bc103d0.ckpt"

sae_cfg = load_config_from_checkpoint(SAE_CHECKPOINT_PATH)
elsa_checkpoint_path = f"{CHECKPOINT_FOLDER}/{DATASET}/{sae_cfg['pretrained_model_checkpoint']}"
elsa_cfg = load_config_from_checkpoint(elsa_checkpoint_path)

interactions_df, train_csr, val_csr, test_csr, train_users, val_users, test_users, items = prepare_interaction_data(elsa_cfg)
train_users_to_idxs = {uid: uidx for uidx, uid in enumerate(train_users)}
val_users_to_idxs = {uid: uidx for uidx, uid in enumerate(val_users)}
test_users_to_idxs = {uid: uidx for uidx, uid in enumerate(test_users)}
items_to_idxs = {iid: iidx for iidx, iid in enumerate(items)}

items_df = (
    pl.scan_csv(f"{DATA_FOLDER}/{DATASET}/movies.csv").rename({"movieId": "item_id"}).cast({"item_id": pl.String}).cast({"item_id": pl.Categorical}).collect()
)

elsa_model_class = getattr(importlib.import_module(elsa_cfg["model_module"]), elsa_cfg["model_class"])
elsa = elsa_model_class(train_csr.shape[1], elsa_cfg["embedding_dim"], elsa_cfg["seed"]).to(device)
_, _ = load_checkpoint(elsa, None, get_checkpoint_filepath(elsa_cfg), device, None)

sae_model_class = getattr(importlib.import_module(sae_cfg["model_module"]), sae_cfg["model_class"])
sae_extra_params = {k: sae_cfg[k] for k in sae_cfg.keys() if k in ["l1_coef", "k"]}
sae = sae_model_class(elsa_cfg["embedding_dim"], sae_cfg["embedding_dim"], sae_cfg["seed"], **sae_extra_params).to(device)
_, _ = load_checkpoint(sae, None, get_checkpoint_filepath(sae_cfg), device, None)

Dataset info: users=162342, items=40858, interactions=12452811
Train split info: users=129874, items=40858, interactions=9961684
Val split info: users=16234, items=40858, interactions=1246445
Test split info: users=16234, items=40858, interactions=1244682
Loaded checkpoint from checkpoints/ml-25m/ELSA-512-991a26a5.ckpt (after 10 epochs)
Loaded checkpoint from checkpoints/ml-25m/TopKSAE-8192-1bc103d0.ckpt (after 99 epochs)


In [2]:
user_id = np.random.choice(train_users)

print("Interactions:")
print(interactions_df.filter(pl.col("user_id") == user_id).join(items_df, on="item_id"))

k = 5
topk_scores, topk_idxs = elsa.recommend(torch.tensor(train_csr[train_users_to_idxs[user_id]].toarray()).to(device), k=5)
topk_scores, topk_idxs = topk_scores.flatten(), topk_idxs.flatten()
topk_item_ids = np.array([items[iidx] for iidx in topk_idxs])

print("Recommendations:")
print(
    pl.DataFrame({"item_id": topk_item_ids, "score": topk_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
)

Interactions:
shape: (33, 5)
┌─────────┬─────────┬───────┬─────────────────────────────────┬─────────────────────────────────┐
│ user_id ┆ item_id ┆ value ┆ title                           ┆ genres                          │
│ ---     ┆ ---     ┆ ---   ┆ ---                             ┆ ---                             │
│ cat     ┆ cat     ┆ f32   ┆ str                             ┆ str                             │
╞═════════╪═════════╪═══════╪═════════════════════════════════╪═════════════════════════════════╡
│ 122307  ┆ 1       ┆ 4.0   ┆ Toy Story (1995)                ┆ Adventure|Animation|Children|C… │
│ 122307  ┆ 11      ┆ 4.0   ┆ American President, The (1995)  ┆ Comedy|Drama|Romance            │
│ 122307  ┆ 17      ┆ 5.0   ┆ Sense and Sensibility (1995)    ┆ Drama|Romance                   │
│ 122307  ┆ 356     ┆ 4.0   ┆ Forrest Gump (1994)             ┆ Comedy|Drama|Romance|War        │
│ 122307  ┆ 450     ┆ 4.0   ┆ With Honors (1994)              ┆ Comedy|Drama             

/tmp/ipykernel_3750102/2276699084.py:4: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  print(interactions_df.filter(pl.col("user_id") == user_id).join(items_df, on="item_id"))
/tmp/ipykernel_3750102/2276699084.py:14: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")


In [3]:
# item_id = np.random.choice(items)
# item_id = "296"  # Pulp Fiction
item_id = "4306"  # Shrek

print("Item:")
print(items_df.filter(pl.col("item_id") == item_id))

k = 5
topk_scores, topk_idxs = elsa.recommend(
    torch.tensor(sp.eye(len(items), dtype=np.float32, format="csr")[items_to_idxs[item_id]].toarray()).to(device).float(), k=5
)
topk_scores, topk_idxs = topk_scores.flatten(), topk_idxs.flatten()
topk_item_ids = np.array([items[iidx] for iidx in topk_idxs])

print("Recommendations:")
print(
    pl.DataFrame({"item_id": topk_item_ids, "score": topk_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
)

Item:
shape: (1, 3)
┌─────────┬──────────────┬─────────────────────────────────┐
│ item_id ┆ title        ┆ genres                          │
│ ---     ┆ ---          ┆ ---                             │
│ cat     ┆ str          ┆ str                             │
╞═════════╪══════════════╪═════════════════════════════════╡
│ 4306    ┆ Shrek (2001) ┆ Adventure|Animation|Children|C… │
└─────────┴──────────────┴─────────────────────────────────┘
Recommendations:
shape: (5, 4)
┌─────────┬──────────┬─────────────────────────────────┬─────────────────────────────────┐
│ item_id ┆ score    ┆ title                           ┆ genres                          │
│ ---     ┆ ---      ┆ ---                             ┆ ---                             │
│ cat     ┆ f32      ┆ str                             ┆ str                             │
╞═════════╪══════════╪═════════════════════════════════╪═════════════════════════════════╡
│ 4886    ┆ 0.921351 ┆ Monsters, Inc. (2001)           ┆ Adventure|

/tmp/ipykernel_3750102/806696857.py:18: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")


In [4]:
user_idx = 123
print("Interactions")
print(interactions_df.filter(pl.col("user_id") == train_users[user_idx]).join(items_df, on="item_id"))

interaction_vector = train_csr[user_idx]  # CSR matrix, shape = (1 x num_items)
interaction_tensor = torch.tensor(interaction_vector.toarray()).to(device)  # Tensor shape = (1 x num_items)

elsa_embedding = elsa.encode(interaction_tensor)  # Tensor, shape = (1 x elsa_cfg["embedding_dim"])

sae_embedding, _, input_mean, input_std = sae.encode(elsa_embedding)  # Tensor, shape = (1 x sae_cfg["embedding_dim"]), sparse (most values are zero)

topk_neuron_values, topk_neuron_ids = torch.topk(sae_embedding, 3)
print(f"Most active neurons: {topk_neuron_ids.cpu().numpy().flatten()} with values {topk_neuron_values.detach().cpu().numpy().flatten()}")

reconstructed_elsa_embedding = sae.decode(sae_embedding, input_mean, input_std)  # Tensor, shape = (1 x elsa_cfg["embedding_dim"])
print(f"Relative reconstruction error norm: {(elsa_embedding - reconstructed_elsa_embedding).norm() / elsa_embedding.norm()}")
print(f"Cosine similarity: {torch.cosine_similarity(elsa_embedding, reconstructed_elsa_embedding).item()}")

elsa_scores = elsa.decode(elsa_embedding)
elsa_scores[interaction_tensor != 0] = 0
topk_elsa_scores, topk_elsa_idxs = torch.topk(elsa_scores, 5)
topk_elsa_scores, topk_elsa_idxs = topk_elsa_scores.detach().cpu().numpy().flatten(), topk_elsa_idxs.cpu().numpy().flatten()
topk_elsa_item_ids = np.array([items[iidx] for iidx in topk_elsa_idxs])
print("Recommendations:")
print(
    pl.DataFrame({"item_id": topk_elsa_item_ids, "score": topk_elsa_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
)

reconstructed_elsa_scores = elsa.decode(reconstructed_elsa_embedding)
reconstructed_elsa_scores[interaction_tensor != 0] = 0
topk_reconstructed_elsa_scores, topk_reconstructed_elsa_idxs = torch.topk(reconstructed_elsa_scores, 5)
topk_reconstructed_elsa_scores, topk_reconstructed_elsa_idxs = (
    topk_reconstructed_elsa_scores.detach().cpu().numpy().flatten(),
    topk_reconstructed_elsa_idxs.cpu().numpy().flatten(),
)
topk_reconstructed_elsa_item_ids = np.array([items[iidx] for iidx in topk_reconstructed_elsa_idxs])
print("Reconstructed Recommendations:")
print(
    pl.DataFrame({"item_id": topk_reconstructed_elsa_item_ids, "score": topk_reconstructed_elsa_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
)

Interactions
shape: (15, 5)
┌─────────┬─────────┬───────┬─────────────────────────────────┬─────────────────────────────────┐
│ user_id ┆ item_id ┆ value ┆ title                           ┆ genres                          │
│ ---     ┆ ---     ┆ ---   ┆ ---                             ┆ ---                             │
│ cat     ┆ cat     ┆ f32   ┆ str                             ┆ str                             │
╞═════════╪═════════╪═══════╪═════════════════════════════════╪═════════════════════════════════╡
│ 118130  ┆ 2       ┆ 4.0   ┆ Jumanji (1995)                  ┆ Adventure|Children|Fantasy      │
│ 118130  ┆ 527     ┆ 4.0   ┆ Schindler's List (1993)         ┆ Drama|War                       │
│ 118130  ┆ 1193    ┆ 5.0   ┆ One Flew Over the Cuckoo's Nes… ┆ Drama                           │
│ 118130  ┆ 1339    ┆ 4.0   ┆ Dracula (Bram Stoker's Dracula… ┆ Fantasy|Horror|Romance|Thrille… │
│ 118130  ┆ 4275    ┆ 4.0   ┆ Krull (1983)                    ┆ Action|Adventure|Fantasy|S

/tmp/ipykernel_3750102/3905893989.py:3: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  print(interactions_df.filter(pl.col("user_id") == train_users[user_idx]).join(items_df, on="item_id"))


Cosine similarity: 0.7689646482467651
Recommendations:
shape: (5, 4)
┌─────────┬──────────┬─────────────────────────────────┬─────────────────────────────┐
│ item_id ┆ score    ┆ title                           ┆ genres                      │
│ ---     ┆ ---      ┆ ---                             ┆ ---                         │
│ cat     ┆ f32      ┆ str                             ┆ str                         │
╞═════════╪══════════╪═════════════════════════════════╪═════════════════════════════╡
│ 858     ┆ 6.322963 ┆ Godfather, The (1972)           ┆ Crime|Drama                 │
│ 593     ┆ 6.167894 ┆ Silence of the Lambs, The (199… ┆ Crime|Horror|Thriller       │
│ 1221    ┆ 6.163879 ┆ Godfather: Part II, The (1974)  ┆ Crime|Drama                 │
│ 318     ┆ 6.015493 ┆ Shawshank Redemption, The (199… ┆ Crime|Drama                 │
│ 296     ┆ 5.627659 ┆ Pulp Fiction (1994)             ┆ Comedy|Crime|Drama|Thriller │
└─────────┴──────────┴─────────────────────────────────┴─────

/tmp/ipykernel_3750102/3905893989.py:27: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")
/tmp/ipykernel_3750102/3905893989.py:42: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")


In [5]:
item_id = "4306"  # Shrek
interaction_tensor = (
    torch.tensor(torch.tensor(sp.eye(len(items), dtype=np.float32, format="csr")[items_to_idxs[item_id]].toarray())).to(device).float()
)  # Tensor shape = (1 x num_items)

elsa_embedding = elsa.encode(interaction_tensor)  # Tensor, shape = (1 x elsa_cfg["embedding_dim"])

sae_embedding, _, input_mean, input_std = sae.encode(elsa_embedding)  # Tensor, shape = (1 x sae_cfg["embedding_dim"]), sparse (most values are zero)

topk_neuron_values, topk_neuron_ids = torch.topk(sae_embedding, 3)
print(f"Most active neurons: {topk_neuron_ids.cpu().numpy().flatten()} with values {topk_neuron_values.detach().cpu().numpy().flatten()}")

reconstructed_elsa_embedding = sae.decode(sae_embedding, input_mean, input_std)  # Tensor, shape = (1 x elsa_cfg["embedding_dim"])
print(f"Relative reconstruction error norm: {(elsa_embedding - reconstructed_elsa_embedding).norm() / elsa_embedding.norm()}")
print(f"Cosine similarity: {torch.cosine_similarity(elsa_embedding, reconstructed_elsa_embedding).item()}")

elsa_scores = elsa.decode(elsa_embedding)
elsa_scores[interaction_tensor != 0] = 0
topk_elsa_scores, topk_elsa_idxs = torch.topk(elsa_scores, 5)
topk_elsa_scores, topk_elsa_idxs = topk_elsa_scores.detach().cpu().numpy().flatten(), topk_elsa_idxs.cpu().numpy().flatten()
topk_elsa_item_ids = np.array([items[iidx] for iidx in topk_elsa_idxs])
print("Recommendations:")
print(
    pl.DataFrame({"item_id": topk_elsa_item_ids, "score": topk_elsa_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
)

reconstructed_elsa_scores = elsa.decode(reconstructed_elsa_embedding)
reconstructed_elsa_scores[interaction_tensor != 0] = 0
topk_reconstructed_elsa_scores, topk_reconstructed_elsa_idxs = torch.topk(reconstructed_elsa_scores, 5)
topk_reconstructed_elsa_scores, topk_reconstructed_elsa_idxs = (
    topk_reconstructed_elsa_scores.detach().cpu().numpy().flatten(),
    topk_reconstructed_elsa_idxs.cpu().numpy().flatten(),
)
topk_reconstructed_elsa_item_ids = np.array([items[iidx] for iidx in topk_reconstructed_elsa_idxs])
print("Reconstructed Recommendations:")
print(
    pl.DataFrame({"item_id": topk_reconstructed_elsa_item_ids, "score": topk_reconstructed_elsa_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
)

Most active neurons: [3003 4671 5653] with values [12.0887165  9.544626   9.3854   ]
Relative reconstruction error norm: 0.7984510064125061
Cosine similarity: 0.9815070629119873
Recommendations:
shape: (5, 4)
┌─────────┬───────────┬─────────────────────────────────┬─────────────────────────────────┐
│ item_id ┆ score     ┆ title                           ┆ genres                          │
│ ---     ┆ ---       ┆ ---                             ┆ ---                             │
│ cat     ┆ f32       ┆ str                             ┆ str                             │
╞═════════╪═══════════╪═════════════════════════════════╪═════════════════════════════════╡
│ 4886    ┆ 20.783836 ┆ Monsters, Inc. (2001)           ┆ Adventure|Animation|Children|C… │
│ 6377    ┆ 20.493134 ┆ Finding Nemo (2003)             ┆ Adventure|Animation|Children|C… │
│ 8360    ┆ 18.043509 ┆ Shrek 2 (2004)                  ┆ Adventure|Animation|Children|C… │
│ 6539    ┆ 17.702894 ┆ Pirates of the Caribbean: The …

/tmp/ipykernel_3750102/1794339784.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(torch.tensor(sp.eye(len(items), dtype=np.float32, format="csr")[items_to_idxs[item_id]].toarray())).to(device).float()
/tmp/ipykernel_3750102/1794339784.py:25: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")
/tmp/ipykernel_3750102/1794339784.py:40: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")


In [6]:
user_idx = 123
print("Interactions")
print(interactions_df.filter(pl.col("user_id") == train_users[user_idx]).join(items_df, on="item_id"))

interaction_vector = train_csr[user_idx]  # CSR matrix, shape = (1 x num_items)
interaction_tensor = torch.tensor(interaction_vector.toarray()).to(device)  # Tensor shape = (1 x num_items)

elsa_embedding = elsa.encode(interaction_tensor)  # Tensor, shape = (1 x elsa_cfg["embedding_dim"])

sae_embedding, _, input_mean, input_std = sae.encode(elsa_embedding)  # Tensor, shape = (1 x sae_cfg["embedding_dim"]), sparse (most values are zero)

boosted_sae_embedding = sae_embedding.clone()
boosted_sae_embedding[0, 4671] += 50
# boosted_sae_embedding[0, 3003] += 10
reconstructed_elsa_embedding = sae.decode(boosted_sae_embedding, input_mean, input_std)
print(f"Relative reconstruction error norm: {(elsa_embedding - reconstructed_elsa_embedding).norm() / elsa_embedding.norm()}")

elsa_scores = elsa.decode(elsa_embedding)
elsa_scores[interaction_tensor != 0] = 0
topk_elsa_scores, topk_elsa_idxs = torch.topk(elsa_scores, 5)
topk_elsa_scores, topk_elsa_idxs = topk_elsa_scores.detach().cpu().numpy().flatten(), topk_elsa_idxs.cpu().numpy().flatten()
topk_elsa_item_ids = np.array([items[iidx] for iidx in topk_elsa_idxs])
print("Recommendations:")
print(
    pl.DataFrame({"item_id": topk_elsa_item_ids, "score": topk_elsa_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
)

reconstructed_elsa_scores = elsa.decode(reconstructed_elsa_embedding)
reconstructed_elsa_scores[interaction_tensor != 0] = 0
topk_reconstructed_elsa_scores, topk_reconstructed_elsa_idxs = torch.topk(reconstructed_elsa_scores, 5)
topk_reconstructed_elsa_scores, topk_reconstructed_elsa_idxs = (
    topk_reconstructed_elsa_scores.detach().cpu().numpy().flatten(),
    topk_reconstructed_elsa_idxs.cpu().numpy().flatten(),
)
topk_reconstructed_elsa_item_ids = np.array([items[iidx] for iidx in topk_reconstructed_elsa_idxs])
print("Reconstructed Recommendations:")
print(
    pl.DataFrame({"item_id": topk_reconstructed_elsa_item_ids, "score": topk_reconstructed_elsa_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
)

Interactions
shape: (15, 5)
┌─────────┬─────────┬───────┬─────────────────────────────────┬─────────────────────────────────┐
│ user_id ┆ item_id ┆ value ┆ title                           ┆ genres                          │
│ ---     ┆ ---     ┆ ---   ┆ ---                             ┆ ---                             │
│ cat     ┆ cat     ┆ f32   ┆ str                             ┆ str                             │
╞═════════╪═════════╪═══════╪═════════════════════════════════╪═════════════════════════════════╡
│ 118130  ┆ 2       ┆ 4.0   ┆ Jumanji (1995)                  ┆ Adventure|Children|Fantasy      │
│ 118130  ┆ 527     ┆ 4.0   ┆ Schindler's List (1993)         ┆ Drama|War                       │
│ 118130  ┆ 1193    ┆ 5.0   ┆ One Flew Over the Cuckoo's Nes… ┆ Drama                           │
│ 118130  ┆ 1339    ┆ 4.0   ┆ Dracula (Bram Stoker's Dracula… ┆ Fantasy|Horror|Romance|Thrille… │
│ 118130  ┆ 4275    ┆ 4.0   ┆ Krull (1983)                    ┆ Action|Adventure|Fantasy|S

/tmp/ipykernel_3750102/1355882819.py:3: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  print(interactions_df.filter(pl.col("user_id") == train_users[user_idx]).join(items_df, on="item_id"))
/tmp/ipykernel_3750102/1355882819.py:26: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")
/tmp/ipykernel_3750102/1355882819.py:41: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")
